Ultralytics 8.3.216  Python-3.9.23 torch-2.8.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 0
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [5]:
!yolo train model=yolo11n.pt data=coco8.yaml epochs=3 imgsz=640

WARNING Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt... <urlopen error TLS/SSL connection has been closed (EOF) (_ssl.c:1147)>
Ultralytics 8.3.216  Python-3.9.23 torch-2.8.0+cpu CPU (13th Gen Intel Core i9-13900HX)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mo


#=#=#                                                                          
##O#-#                                                                         
##O=#  #                                                                       
#=#=-#  #                                                                      
-#O#- #   #                                                                    
-=#=#   #   #                                                                  
-=O#-#   #   #                                                                 
-=O=#  #   #   #                                                               
-=O=-#  #    #   #                                                             
                                                                           0.5%
                                                                           0.8%
#                                                                          1.4%
#                                      

In [ ]:
from ultralytics import YOLO
import torch
from ultralytics.nn.modules import Bottleneck, Conv, C2f, SPPF, Detect, C3k2
from torch.nn.modules.container import Sequential
import os
 
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"
 
class PRUNE():
    def __init__(self) -> None:
        self.threshold = None
 
    def get_threshold(self, model, factor=0.8):
        ws = []
        bs = []
        for name, m in model.named_modules():
            if isinstance(m, torch.nn.BatchNorm2d):
                w = m.weight.abs().detach()
                b = m.bias.abs().detach()
                ws.append(w)
                bs.append(b)
                print(name, w.max().item(), w.min().item(), b.max().item(), b.min().item())
 
        # keep
        ws = torch.cat(ws)
        self.threshold = torch.sort(ws, descending=True)[0][int(len(ws) * factor)]
 
    def prune_conv(self, conv1: Conv, conv2: Conv):
        ## Normal Pruning
        gamma = conv1.bn.weight.data.detach()
        beta = conv1.bn.bias.data.detach()
 
        keep_idxs = []
        local_threshold = self.threshold
        while len(keep_idxs) < 8:  ## 若剩余卷积核<8, 则降低阈值重新筛选
            keep_idxs = torch.where(gamma.abs() >= local_threshold)[0]
            local_threshold = local_threshold * 0.5
        n = len(keep_idxs)
        # n = max(int(len(idxs) * 0.8), p)
        print(n / len(gamma) * 100)
        conv1.bn.weight.data = gamma[keep_idxs]
        conv1.bn.bias.data = beta[keep_idxs]
        conv1.bn.running_var.data = conv1.bn.running_var.data[keep_idxs]
        conv1.bn.running_mean.data = conv1.bn.running_mean.data[keep_idxs]
        conv1.bn.num_features = n
        conv1.conv.weight.data = conv1.conv.weight.data[keep_idxs]
        conv1.conv.out_channels = n
 
        if isinstance(conv2, list) and len(conv2) > 3 and conv2[-1]._get_name() == "Proto":
            proto = conv2.pop()
            proto.cv1.conv.in_channels = n
            proto.cv1.conv.weight.data = proto.cv1.conv.weight.data[:, keep_idxs]
        if conv1.conv.bias is not None:
            conv1.conv.bias.data = conv1.conv.bias.data[keep_idxs]
 
        ## Regular Pruning
        if not isinstance(conv2, list):
            conv2 = [conv2]
        for item in conv2:
            if item is None: continue
            if isinstance(item, Conv):
                conv = item.conv
            else:
                conv = item
            if isinstance(item, Sequential):
                conv1 = item[0]
                conv = item[1].conv
                conv1.conv.in_channels = n
                conv1.conv.out_channels = n
                conv1.conv.groups = n
                conv1.conv.weight.data = conv1.conv.weight.data[keep_idxs, :]
                conv1.bn.bias.data = conv1.bn.bias.data[keep_idxs]
                conv1.bn.weight.data = conv1.bn.weight.data[keep_idxs]
                conv1.bn.running_var.data = conv1.bn.running_var.data[keep_idxs]
                conv1.bn.running_mean.data = conv1.bn.running_mean.data[keep_idxs]
                conv1.bn.num_features = n
            conv.in_channels = n
            conv.weight.data = conv.weight.data[:, keep_idxs]
 
    def prune(self, m1, m2):
        if isinstance(m1, C3k2):  # C3k2 as a top conv
            m1 = m1.cv2
        if isinstance(m1, Sequential):
            m1 = m1[1]
        if not isinstance(m2, list):  # m2 is just one module
            m2 = [m2]
        for i, item in enumerate(m2):
            if isinstance(item, C3k2) or isinstance(item, SPPF):
                m2[i] = item.cv1
 
        self.prune_conv(m1, m2)
 
 
def do_pruning(modelpath, savepath):
    pruning = PRUNE()
 
    ### 0. 加载模型
    yolo = YOLO(modelpath)  # build a new model from scratch
    pruning.get_threshold(yolo.model, 0.8)  # 这里的0.8为剪枝率。
 
    ### 1. 剪枝C3k2 中的Bottleneck
    for name, m in yolo.model.named_modules():
        if isinstance(m, Bottleneck):
            pruning.prune_conv(m.cv1, m.cv2)
 
    ### 2. 指定剪枝不同模块之间的卷积核
    seq = yolo.model.model
    for i in [3, 5, 7, 8]:
        pruning.prune(seq[i], seq[i + 1])
 
    ### 3. 对检测头进行剪枝
    # 在P3层: seq[15]之后的网络节点与其相连的有 seq[16]、detect.cv2[0] (box分支)、detect.cv3[0] (class分支)
    # 在P4层: seq[18]之后的网络节点与其相连的有 seq[19]、detect.cv2[1] 、detect.cv3[1]
    # 在P5层: seq[21]之后的网络节点与其相连的有 detect.cv2[2] 、detect.cv3[2]
    detect: Detect = seq[-1]
    # proto = detect.proto
    last_inputs = [seq[16], seq[19], seq[22]]
    colasts = [seq[17], seq[20], None]
    for idx, (last_input, colast, cv2, cv3) in enumerate(zip(last_inputs, colasts, detect.cv2, detect.cv3)):
        if idx == 0:
            pruning.prune(last_input, [colast, cv2[0], cv3[0]])
        else:
            pruning.prune(last_input, [colast, cv2[0], cv3[0]])
        pruning.prune(cv2[0], cv2[1])
        pruning.prune(cv2[1], cv2[2])
        pruning.prune(cv3[0], cv3[1])
        pruning.prune(cv3[1], cv3[2])
 
    ### 4. 模型梯度设置与保存
    for name, p in yolo.model.named_parameters():
        p.requires_grad = True
 
    yolo.info()
 
    yolo.val(data='coco8.yaml', batch=16, device=0, workers=0)
    torch.save(yolo.ckpt, savepath)
 
if __name__ == "__main__":
    modelpath = r"runs/detect/train/weights/last.pt"
    savepath = r"runs/detect/train/weights/last_prune.pt"
    do_pruning(modelpath, savepath)

model.0.bn 9.8125 1.6650390625 12.7578125 0.10186767578125
model.1.bn 5.19140625 2.37890625 4.91796875 0.0038242340087890625
model.2.cv1.bn 5.41015625 1.435546875 3.666015625 0.03167724609375
model.2.cv2.bn 4.73828125 0.728515625 4.359375 0.0169219970703125
model.2.m.0.cv1.bn 1.38671875 0.2291259765625 1.2666015625 0.57666015625
model.2.m.0.cv2.bn 3.61328125 2.017578125 9.4375 0.342041015625
model.3.bn 1.5361328125 0.55908203125 2.611328125 0.00794219970703125
model.4.cv1.bn 1.736328125 0.313232421875 1.73828125 0.02099609375
model.4.cv2.bn 1.841796875 0.20361328125 2.626953125 0.0244293212890625
model.4.m.0.cv1.bn 0.787109375 0.293701171875 1.4765625 0.017913818359375
model.4.m.0.cv2.bn 1.1748046875 0.50927734375 3.060546875 0.047515869140625
model.5.bn 1.1376953125 0.34228515625 2.099609375 0.0097808837890625
model.6.cv1.bn 2.81640625 0.465576171875 3.1171875 0.00484466552734375
model.6.cv2.bn 1.2744140625 0.383544921875 2.26171875 0.033172607421875
model.6.m.0.cv1.bn 0.912109375 0.3

FileNotFoundError: 'own.yaml' does not exist